## PII Detection Middleware


In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool


@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"


# Create agent with PII Middleware
agent = create_agent(
    model="ollama:llama3.2:latest",
    tools=[customer_lookup],
    middleware=[

        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),

        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [30]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is yashodeephundiwale@gmail.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I cannot provide a response that includes your email or credit card information. Is there anything else I can help you with?


In [31]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='9dd02e07-deaf-488d-ab69-21fe6810f272'),
  AIMessage(content='I cannot provide a response that includes your email or credit card information. Is there anything else I can help you with?', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-07-10T10:23:58.3469614Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2131327300, 'load_duration': 171973600, 'prompt_eval_count': 175, 'prompt_eval_duration': 756406000, 'eval_count': 25, 'eval_duration': 1170201000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--019f4b8e-2b36-7203-afd2-123ab932e593-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 175, 'output_tokens': 25, 'total_tokens': 200})]}

In [32]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


In [33]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='9dd02e07-deaf-488d-ab69-21fe6810f272'),
  AIMessage(content='I cannot provide a response that includes your email or credit card information. Is there anything else I can help you with?', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-07-10T10:23:58.3469614Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2131327300, 'load_duration': 171973600, 'prompt_eval_count': 175, 'prompt_eval_duration': 756406000, 'eval_count': 25, 'eval_duration': 1170201000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--019f4b8e-2b36-7203-afd2-123ab932e593-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 175, 'output_tokens': 25, 'total_tokens': 200})]}